**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Training Dynamics

The craft between [ANN's](./Intro_ANN/Intro_ANN.ipynb) backprop and [Scale_NN's](./Scale_NN/Scale_NN.ipynb) systems view: which optimizer, what schedule, how much regularization — with every claim demonstrated by an experiment you can rerun.

## 1. Pre-requisites

- [Intro to ANN](./Intro_ANN/Intro_ANN.ipynb) and [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) — condition numbers and SGD noise floors.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# Standard testbed for the whole workshop: noisy spiral classification (from the ANN workshop)
def spirals(n=1500, noise=0.35, seed=0):
    r = np.random.default_rng(seed)
    t = np.linspace(0.5, 3*np.pi, n//2)
    X, y = [], []
    for cls, ph in [(0, 0.0), (1, np.pi)]:
        X.append(np.stack([t*np.cos(t+ph), t*np.sin(t+ph)], 1) + noise*r.standard_normal((n//2, 2)))
        y.append(np.full(n//2, cls))
    X = np.concatenate(X).astype(np.float32); y = np.concatenate(y)
    X = (X - X.mean(0)) / X.std(0)
    idx = r.permutation(n)
    return map(torch.from_numpy, (X[idx][:1000], y[idx][:1000].astype(np.int64),
                                  X[idx][1000:], y[idx][1000:].astype(np.int64)))
Xtr, ytr, Xte, yte = spirals()

def make_model(width=64):
    torch.manual_seed(1)
    return nn.Sequential(nn.Linear(2, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 2))

def train(model, opt, epochs=150, sched=None, wd_manual=0.0):
    lossf = nn.CrossEntropyLoss(); hist = []
    for ep in range(epochs):
        for i in range(0, 1000, 100):
            xb, yb = Xtr[i:i+100], ytr[i:i+100]
            opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
        if sched: sched.step()
        with torch.no_grad():
            hist.append(( lossf(model(Xtr), ytr).item(),
                          (model(Xte).argmax(1) == yte).float().mean().item() ))
    return np.array(hist)

---
### 🕐 Session 1 of 3 — *Optimizers: SGD → Momentum → Adam* (~40 min)
**Goal:** understand what each optimizer adds, and race them fairly.
**Builds on:** [Optimization](../Intro_Math/Optimization/Optimization.ipynb) S2/S4. &nbsp; **Feeds into:** Session 2 (schedules).

---

## 2. The Optimizer Ladder

💡 **Intuition.** **Momentum** treats the gradient as a force, not a velocity: updates accumulate, so consistent directions build speed while zigzag components ([Optimization's](../Intro_Math/Optimization/Optimization.ipynb) κ-canyon!) cancel — a heavy ball rolling through the noise. **Adam** adds a per-parameter yardstick: divide each coordinate's step by its own recent gradient RMS, so rarely-updated weights take bold steps and jittery ones take timid steps — approximate per-axis preconditioning. Neither is magic: they attack curvature and scale imbalance, nothing else.

In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 2 of 3 — *Learning-Rate Schedules & Warmup* (~35 min)
**Goal:** big steps to travel, small steps to settle — and why transformers need warmup.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (regularization).

---

## 3. Schedules

💡 **Intuition.** [Optimization S4](../Intro_Math/Optimization/Optimization.ipynb) proved the constant-step noise floor: finishing requires *decay*. Cosine decay is the popular smooth path from exploration to settlement. **Warmup** (starting tiny and ramping up) protects the fragile early phase — at initialization, curvature estimates (Adam's RMS yardsticks) are garbage computed from a handful of noisy batches; a few gentle epochs let the statistics stabilize before taking real steps. Deep transformers can *diverge irrecoverably* without it.

In [ ]:

# YOUR CODE HERE


**Honest reading:** on this small, low-noise task all three schedules land in the same place — the noise floor that decay exists to fix is tiny here (you *saw* it clearly in [Optimization S4](../Intro_Math/Optimization/Optimization.ipynb)'s SGD experiment, where gradient noise was substantial). The ranking flips at scale: large batches of noisy data, deep transformers, and long horizons are where cosine + warmup stop being optional. Treat schedules as insurance whose premium is zero — the standard recipe (warmup → cosine) never hurts and sometimes rescues the run.

---
### 🕐 Session 3 of 3 — *Regularization & Generalization* (~40 min)
**Goal:** weight decay, dropout, early stopping — and a live look at the fit/overfit boundary.
**Builds on:** Session 2.

---

## 4. Fighting Memorization

💡 **Intuition.** A big network *can* memorize noise; regularization is a thumb on the scale toward simple functions. **Weight decay** shrinks weights each step (an L2 penalty — small weights ⇒ smoother functions). **Dropout** randomly silences units during training, forcing redundancy — an implicit [ensemble](./Uncertainty_in_ML.ipynb). **Early stopping** just quits while the *validation* loss is still improving — regularization by impatience, and the cheapest of the three. The diagnostic that rules them all: the gap between train and validation curves.

In [ ]:
# Overfit on purpose (small noisy data, big net), then regularize three ways

# YOUR CODE HERE


**Honest reading of the table:** on a 120-sample toy problem the differences are real but modest — dropout helps most here, and early stopping alone recovers most of the benefit for free. Regularization's value *scales with the memorization opportunity*; rerun with `spirals(noise=0.5)` and width 512 to watch the gaps widen.

**On double descent** (stated, worth knowing): in modern regimes, pushing model size *past* the interpolation point can make test error fall *again* — the classical U-curve is incomplete for very large models. The practical takeaways survive: monitor the train/val gap, and when in doubt, more data beats more tricks.

## 5. Conclusion

Momentum cancels zigzag, Adam equalizes axes, schedules trade exploration for settlement, warmup protects fragile statistics, and regularization is a thumb on the scale toward simplicity — each one an experiment you just ran, not a slogan.

---
## Where next

- [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb) — the systems half of training.
- [Uncertainty in ML](./Uncertainty_in_ML.ipynb) — what the val-gap means for trust.
- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — these recipes at their most extreme.